# Generate stops.txt

Generates GTFS stops file with equidistant stops along each route shape.

In [1]:
import json
from pathlib import Path 
import pandas as pd
import geopandas as gpd


## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]
AGENCY_ID = p["agency"]["id"]
distance_between_stops = p["stops"]["distance_between_stops"]

rutas_entrada

In [4]:
# --- GTFS Folder ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_GTFS.absolute()}")

# --- Processed Folder ---
PATH_DIR_proccesed = Path(f"../data/{CITY}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_proccesed.absolute()}")

Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Lectura de archivos

In [5]:
routes_clean_path =  PATH_DIR_proccesed / "routes_clean.geojson"

routes_clean = gpd.read_file(routes_clean_path)
routes_clean.head()

,route_name,route_name_short,route_type,agency_id,shape_id,geometry
0,2_42 Caseta,42 Caseta,3,Red_de_Transporte_Merida,Shape_2_42 Caseta,"LINESTRING (1478265.739 2346761.139, 1478081.3..."
1,6_42 Sur Imss,42 Sur Imss,3,Red_de_Transporte_Merida,Shape_6_42 Sur Imss,"LINESTRING (1478238.108 2346765.115, 1478079.2..."
2,11_50 Penal Paso Texas,50 Penal Paso Texas,3,Red_de_Transporte_Merida,Shape_11_50 Penal Paso Texas,"LINESTRING (1478226.696 2346744.249, 1478231.8..."
3,13_50 Sur Villa Magna,50 Sur Villa Magna,3,Red_de_Transporte_Merida,Shape_13_50 Sur Villa Magna,"LINESTRING (1478011.172 2346914.365, 1478097.8..."
4,16_Zazil-Ha,Zazil-Ha,3,Red_de_Transporte_Merida,Shape_16_Zazil-Ha,"LINESTRING (1478222.833 2346720.463, 1478230.8..."


In [6]:
# Asegúrate de que está en 4326 (grados)
print(routes_clean.crs)  # debería decir EPSG:4326
# Reproyectar a UTM 14N (metros)
routes_m = routes_clean.to_crs(epsg=32614)

# Calcular longitud en metros
routes_m["length_m"] = routes_m.geometry.length

routes_m[["route_name", "route_name_short", "length_m"]].head()

EPSG:32614


,route_name,route_name_short,length_m
0,2_42 Caseta,42 Caseta,10944.623928
1,6_42 Sur Imss,42 Sur Imss,11318.811971
2,11_50 Penal Paso Texas,50 Penal Paso Texas,18612.678502
3,13_50 Sur Villa Magna,50 Sur Villa Magna,13005.158385
4,16_Zazil-Ha,Zazil-Ha,18273.743274


## Generar stops equidistantes para cada ruta y segmentos
Toma el linestring y lo divide en N segmentos

In [7]:
# Segmentos de longitud fija (metros). Stops en los extremos de cada segmento.
# Puedes usar distance_between_stops = 22 para paradas cada 22 m, o 200 para cada 200 m.
stops_rows = []
segments_rows = []

In [8]:
from shapely.ops import substring

def _line_substring(line, d0, d1):
    """Extrae el tramo exacto siguiendo las curvas de la línea."""
    return substring(line, d0, d1, normalized=False)

In [9]:
for idx, row in routes_m.iterrows():
    route_name = row["route_name"]
    route_name_short = row["route_name_short"]
    shape_id = f"shape_{route_name_short}"
    line = row["geometry"]
    if line is None or line.is_empty:
        continue
    total_m = line.length

    # #endregion
    n_segments = max(1, int(total_m // distance_between_stops))
    # Stops en 0, distance_between_stops, 2*distance_between_stops, ... y el último en min(n_segments * distance_between_stops, total_m)
    for stop_seq in range(n_segments + 1):
        measure_m = min(stop_seq * distance_between_stops, total_m)
        point = line.interpolate(measure_m)
        stop_id = f"{route_name}_{stop_seq:04d}"
        stops_rows.append({
            "route_name": route_name,
            "route_name_short": route_name_short,
            "shape_id": shape_id,
            "stop_seq": stop_seq,
            "measure_m": float(measure_m),
            "stop_id": stop_id,
            "geometry": point,
        })
    # Segmentos entre stops consecutivos
    for seg_seq in range(n_segments):
        from_m = seg_seq * distance_between_stops
        to_m = min((seg_seq + 1) * distance_between_stops, total_m)
        seg_geom = _line_substring(line, from_m, to_m)
        from_stop_id = f"{route_name}_{seg_seq:04d}"
        to_stop_id = f"{route_name}_{seg_seq + 1:04d}"
        segment_id = f"Seg_{route_name}_{seg_seq:04d}"
        length_m = to_m - from_m
        segments_rows.append({
            "route_name": route_name,
            "shape_id": shape_id,
            "segment_seq": seg_seq,
            "segment_id": segment_id,
            "from_stop_id": from_stop_id,
            "to_stop_id": to_stop_id,
            "from_measure_m": float(from_m),
            "to_measure_m": float(to_m),
            "length_m": float(length_m),
            "geometry": seg_geom,
        })

In [10]:

gdf_stops = gpd.GeoDataFrame(stops_rows, geometry="geometry", crs=routes_m.crs)
gdf_segments = gpd.GeoDataFrame(segments_rows, geometry="geometry", crs=routes_m.crs)

# Opcional: volver a WGS84 para uso en GTFS
gdf_stops = gdf_stops.to_crs(4326)
gdf_segments = gdf_segments.to_crs(4326)

# Formato tablas: columnas en el orden solicitado
gdf_stops = gdf_stops[["route_name", "route_name_short", "shape_id", "stop_seq", "measure_m", "stop_id", "geometry"]]
gdf_segments = gdf_segments[["route_name", "shape_id", "segment_seq", "segment_id", "from_stop_id", "to_stop_id",
                             "from_measure_m", "to_measure_m",  "geometry"]]




In [11]:
gdf_segments.head(3)

,route_name,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry
0,2_42 Caseta,shape_42 Caseta,0,Seg_2_42 Caseta_0000,2_42 Caseta_0000,2_42 Caseta_0001,0.0,200.0,"LINESTRING (-89.62102 20.96196, -89.62276 20.9..."
1,2_42 Caseta,shape_42 Caseta,1,Seg_2_42 Caseta_0001,2_42 Caseta_0001,2_42 Caseta_0002,200.0,400.0,"LINESTRING (-89.62278 20.96209, -89.62295 20.9..."
2,2_42 Caseta,shape_42 Caseta,2,Seg_2_42 Caseta_0002,2_42 Caseta_0002,2_42 Caseta_0003,400.0,600.0,"LINESTRING (-89.62201 20.96102, -89.62162 20.9..."


In [12]:
gdf_stops.head(3)

,route_name,route_name_short,shape_id,stop_seq,measure_m,stop_id,geometry
0,2_42 Caseta,42 Caseta,shape_42 Caseta,0,0.0,2_42 Caseta_0000,POINT (-89.62102 20.96196)
1,2_42 Caseta,42 Caseta,shape_42 Caseta,1,200.0,2_42 Caseta_0001,POINT (-89.62278 20.96209)
2,2_42 Caseta,42 Caseta,shape_42 Caseta,2,400.0,2_42 Caseta_0002,POINT (-89.62201 20.96102)


## Format stops to GTFS format

In [13]:
# Convert gdf_stops to proper GTFS format
stops_gtfs_rows = []
for idx, row in gdf_stops.iterrows():
    point = row["geometry"]
    stops_gtfs_rows.append({
        "stop_id": row["stop_id"],
        "stop_code": "",
        "stop_name": f"Stop {row['stop_seq']} - {row['route_name']}",
        "stop_lat": point.y,
        "stop_lon": point.x,
        "location_type": "0",
        "parent_station": "",
        "wheelchair_boarding": "0",
    })

stops_gtfs = pd.DataFrame(stops_gtfs_rows)

In [14]:
# GTFS format stops created from gdf_stops in the previous cell

In [15]:
stops_gtfs.tail()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding
6450,F15_San Pedro Noh-Pat R2_0116,,Stop 116 - F15_San Pedro Noh-Pat R2,20.963636,-89.618463,0,,0
6451,F15_San Pedro Noh-Pat R2_0117,,Stop 117 - F15_San Pedro Noh-Pat R2,20.964286,-89.619580,0,,0
6452,F15_San Pedro Noh-Pat R2_0118,,Stop 118 - F15_San Pedro Noh-Pat R2,20.962754,-89.619637,0,,0
6453,F15_San Pedro Noh-Pat R2_0119,,Stop 119 - F15_San Pedro Noh-Pat R2,20.962200,-89.617830,0,,0
6454,F15_San Pedro Noh-Pat R2_0120,,Stop 120 - F15_San Pedro Noh-Pat R2,20.961646,-89.616022,0,,0


In [16]:
total_m = line.length

# #endregion
n_segments = max(1, int(total_m // distance_between_stops))
# Stops at 0, distance_between_stops, 2*distance_between_stops, ... and the last at min(n_segments * distance_between_stops, total_m)
for stop_seq in range(n_segments + 1):
    measure_m = min(stop_seq * distance_between_stops, total_m)
    point = line.interpolate(measure_m)
    stop_id = f"{route_name}_{stop_seq:04d}"
    stops_rows.append({
        "stop_id": stop_id,
        "stop_code": "",
        "stop_name": f"Stop {stop_seq} - {route_name}",
        "stop_lat": point.y,
        "stop_lon": point.x,
        "location_type": "0",
        "parent_station": "",
        "wheelchair_boarding": "0",
    })
# Segments between consecutive stops
for seg_seq in range(n_segments):
    from_m = seg_seq * distance_between_stops
    to_m = min((seg_seq + 1) * distance_between_stops, total_m)
    seg_geom = _line_substring(line, from_m, to_m)
    from_stop_id = f"{route_name}_{seg_seq:04d}"
    to_stop_id = f"{route_name}_{seg_seq + 1:04d}"

## Export files

In [17]:
stops_gtfs.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding
0,2_42 Caseta_0000,,Stop 0 - 2_42 Caseta,20.961960,-89.621020,0,,0
1,2_42 Caseta_0001,,Stop 1 - 2_42 Caseta,20.962091,-89.622785,0,,0
2,2_42 Caseta_0002,,Stop 2 - 2_42 Caseta,20.961017,-89.622007,0,,0
3,2_42 Caseta_0003,,Stop 3 - 2_42 Caseta,20.960511,-89.620184,0,,0
4,2_42 Caseta_0004,,Stop 4 - 2_42 Caseta,20.959982,-89.618368,0,,0


In [18]:
gdf_segments.head()

,route_name,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry
0,2_42 Caseta,shape_42 Caseta,0,Seg_2_42 Caseta_0000,2_42 Caseta_0000,2_42 Caseta_0001,0.0,200.0,"LINESTRING (-89.62102 20.96196, -89.62276 20.9..."
1,2_42 Caseta,shape_42 Caseta,1,Seg_2_42 Caseta_0001,2_42 Caseta_0001,2_42 Caseta_0002,200.0,400.0,"LINESTRING (-89.62278 20.96209, -89.62295 20.9..."
2,2_42 Caseta,shape_42 Caseta,2,Seg_2_42 Caseta_0002,2_42 Caseta_0002,2_42 Caseta_0003,400.0,600.0,"LINESTRING (-89.62201 20.96102, -89.62162 20.9..."
3,2_42 Caseta,shape_42 Caseta,3,Seg_2_42 Caseta_0003,2_42 Caseta_0003,2_42 Caseta_0004,600.0,800.0,"LINESTRING (-89.62018 20.96051, -89.61837 20.9..."
4,2_42 Caseta,shape_42 Caseta,4,Seg_2_42 Caseta_0004,2_42 Caseta_0004,2_42 Caseta_0005,800.0,1000.0,"LINESTRING (-89.61837 20.95998, -89.61699 20.9..."


In [19]:
# archivos procesamiento
gdf_segments.to_file(PATH_DIR_proccesed / "segments.geojson", driver="GeoJSON")
gdf_stops.to_file(PATH_DIR_proccesed / "stops.geojson", driver="GeoJSON")


In [20]:
stops_gtfs.to_csv(PATH_DIR_GTFS / "stops.txt", index=False)